# Camillion — Single-Symbol Baseline (gravity only + 5m open-gate)

**The clean first experiment.** Edit only the **CONFIG** cell (mainly `SYMBOL`), then **Runtime -> Run all**. Gravity-only, one real 1m CSV, train/test split, one honest out-of-sample number. A 5m-CCI gate blocks new opens when the short-term market is flat.

In [ ]:
# ===== SETUP (run this first) =====
import os, sys, subprocess
OWNER, REPO = "monty313", "Camillion_Claude_RL_model"
# Private repo -> read a GitHub token from Colab's Secrets panel (the key icon on the
# left). Add a secret named GITHUB_TOKEN = your ghp_... token, and enable notebook access.
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{OWNER}/{REPO}.git"
if not os.path.isdir(REPO) and os.path.basename(os.getcwd()) != REPO:
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
subprocess.run(["git", "pull", "--quiet", "--ff-only"], check=False)
print("installing deps (~1-2 min the first time)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "stable-baselines3[extra]", "gymnasium", "torch", "pandas", "numpy"], check=True)
from google.colab import drive
drive.mount("/content/drive")
print("setup done - repo cloned, Drive mounted.")

In [ ]:
# ============================ EDIT ME (mostly SYMBOL) ============================
SYMBOL          = "EURUSD"
CSV_PATH        = ""             # leave "" to AUTO-FIND your EURUSD file on Drive
MAX_BARS        = 300_000        # most-recent N 1m bars for speed (None = all ~2M)
POSITION_SIZE   = 100000.0
REWARD_SCALE    = 1.0
TOTAL_TIMESTEPS = 20_000         # SMOKE TEST first; then bump to 500_000
WARMUP          = 260
WINDOW          = 5_000
TRAIN_FRACTION  = 0.80
OPEN_GATE_5M    = True           # block new opens when EITHER 5m CCI (30 or 100) in [-50,50]
SAVE_PATH       = f"/content/drive/MyDrive/camillion_single_{SYMBOL}"
# ================================================================================

In [ ]:
# ===== FIND CSV -> LOAD (MT5-robust) -> SPLIT -> TRAIN -> OOS VERDICT =====
import numpy as np, pandas as pd, glob
from src.data.cache_builder import build_aligned_indicators
from src.strategies.registry import AlphaRegistry
from src.strategies.alpha_pack import register_all
from src.training.trainer import train, load_for_eval
from config.ftmo_config import load_active_config

def reg_factory():
    r = AlphaRegistry(); register_gravity_alpha(r); return r

def load_csv_robust(path):
    """Handles MT5 exports (tab-sep, <DATE>/<TIME> split) and plain CSVs alike."""
    raw = pd.read_csv(path, sep=None, engine="python")          # auto-detect , ; or tab
    low = {c.lower().strip(): c for c in raw.columns}
    def pick(*names):
        for n in names:
            if n in low: return low[n]
        return None
    dc, tc = pick("<date>", "date"), pick("<time>", "time")
    dtc = pick("datetime", "date_time", "timestamp", "gmt time", "gmt_time")
    if dc and tc and dc != tc:
        idx = pd.to_datetime(raw[dc].astype(str) + " " + raw[tc].astype(str), errors="coerce")
    elif dtc:
        idx = pd.to_datetime(raw[dtc], errors="coerce")
    else:
        idx = pd.to_datetime(raw[dc], errors="coerce")
    def col(*names):
        c = pick(*names); return pd.to_numeric(raw[c], errors="coerce").to_numpy() if c else None
    o, h, l, c = col("<open>","open"), col("<high>","high"), col("<low>","low"), col("<close>","close")
    v = col("<tickvol>","tickvol","<vol>","vol","volume")
    out = pd.DataFrame({"open": o, "high": h, "low": l, "close": c,
                        "volume": v if v is not None else np.ones(len(raw))},
                       index=pd.DatetimeIndex(np.asarray(idx)))
    out = out[~out.index.isna()].dropna(subset=["open","high","low","close"])
    return out[~out.index.duplicated(keep="last")].sort_index()

if not CSV_PATH:
    hits = sorted(set(glob.glob(f"/content/drive/MyDrive/**/{SYMBOL}_M1_*.csv", recursive=True)
                    + glob.glob(f"/content/drive/MyDrive/**/{SYMBOL}*1m*.csv", recursive=True)
                    + glob.glob(f"/content/drive/MyDrive/**/{SYMBOL}*.csv", recursive=True)))
    assert hits, f"No {SYMBOL} CSV found on Drive. Set CSV_PATH manually."
    CSV_PATH = hits[0]
print("data file:", CSV_PATH)

df = load_csv_robust(CSV_PATH)
if MAX_BARS:
    df = df.iloc[-MAX_BARS:]
print("loaded", f"{len(df):,}", "bars", df.index.min(), "->", df.index.max())
ind = build_aligned_indicators(df)
close = df["close"].values.astype("float32")
time_ns = df.index.values.astype("datetime64[ns]").astype("int64")
T = len(close); split = int(T * TRAIN_FRACTION)
print(f"{SYMBOL}: {T:,} bars -> train {split:,} | test {T-split:,} | 5m gate {OPEN_GATE_5M}")

model = train(ind[:split], close[:split], time_ns[:split], reg_factory,
              total_timesteps=TOTAL_TIMESTEPS, n_envs=8, save_path=SAVE_PATH,
              position_size=POSITION_SIZE, reward_scale=REWARD_SCALE, warmup=WARMUP,
              window=WINDOW, random_window=True, open_gate=OPEN_GATE_5M)
print("trained; saved to", SAVE_PATH)

lo = max(0, split - WARMUP)
model, venv = load_for_eval(SAVE_PATH, ind[lo:], close[lo:], time_ns[lo:], reg_factory,
                            position_size=POSITION_SIZE, warmup=WARMUP, open_gate=OPEN_GATE_5M)
cfg = load_active_config(); start = cfg.starting_balance
obs = venv.reset(); eq = []
for _ in range(len(close[lo:]) - WARMUP - 2):
    a, _ = model.predict(obs, deterministic=True)
    obs, r, dones, infos = venv.step(a); eq.append(infos[0]["equity"])
    if bool(np.asarray(dones).any()):
        break
eq = np.array(eq, float)
if len(eq):
    ret = (eq[-1] - start) / start * 100
    peak = np.maximum.accumulate(eq); maxdd = ((peak - eq) / peak).max() * 100
    wall = float(getattr(cfg, "trailing_drawdown_pct", 4.0))
    print("=" * 56)
    print(f"  OUT-OF-SAMPLE {SYMBOL}:  return {ret:+.2f}%   maxDD {maxdd:.2f}%   bars {len(eq):,}")
    print(f"  4% wall -> {'BREACHED' if maxdd >= wall else 'survived'}")
    print("=" * 56)
else:
    print("no eval steps; lower WARMUP or raise MAX_BARS")

In [ ]:
# ===== DIAGNOSTIC: which alphas actually predict direction on YOUR data? =====
import numpy as np
from src.env.trading_env import TradingEnv
from src.strategies.registry import AlphaRegistry
from src.strategies.alpha_pack import register_all
from src.barbershop.alpha_edge import per_alpha_edge, edge_table
reg2 = AlphaRegistry(); register_all(reg2)
lo = max(0, split - WARMUP)
diag = TradingEnv(ind[lo:], close[lo:].astype("float32"), time_ns[lo:], reg2, warmup=WARMUP)
rep = per_alpha_edge(diag.alpha_matrix[:, :15], close[lo:].astype(float), horizon=240)  # 4h ahead
names = [reg2._slots[i].name if reg2._slots[i] else "" for i in range(15)]
print(edge_table(rep, names))
print("\nedge(bps) > 0 AND hit% > 50 -> the alpha predicts direction. Near zero -> noise (cut candidate).")